# FASE 2B: AutoML Benchmark (LazyPredict)**Proyecto Integrador — Grupo 2:** Análisis de Satisfacción en Productos de Oficina**Objetivo:** Benchmark de 27+ clasificadores con LazyPredict + modelo explicativo LightGBM

In [ ]:
# Instalacion de dependencias!pip install mlflow lazypredict -q

## Configuración del Entorno

In [ ]:
import osimport sysimport gcimport jsonfrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport joblibimport mlflowfrom sklearn.model_selection import train_test_splitfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.preprocessing import LabelEncoderfrom sklearn.pipeline import Pipelinefrom sklearn.metrics import f1_scorefrom lazypredict.Supervised import LazyClassifierimport lightgbm as lgbfrom tqdm.auto import tqdmRANDOM_SEED = 42

In [ ]:
IN_COLAB = 'google.colab' in sys.modulesHAS_CUDA = FalseHAS_CUDF = FalseHAS_TORCH = FalseGPU_MEMORY = 0TOTAL_MEMORY = 0try:    import torch    HAS_TORCH = True    HAS_CUDA = torch.cuda.is_available()    if HAS_CUDA:        GPU_MEMORY = torch.cuda.get_device_properties(0).total_memoryexcept ImportError:    passtry:    import cudf    HAS_CUDF = Trueexcept ImportError:    passtry:    import psutil    TOTAL_MEMORY = psutil.virtual_memory().totalexcept ImportError:    passif IN_COLAB:    from google.colab import drive    drive.mount('/content/drive')    BASE = Path('/content/drive/MyDrive/ML/proyecto_integrador')else:    BASE = Path('..')DATA_DIR = BASE / 'data'REPORTS_DIR = BASE / 'reports'MODELS_DIR = BASE / 'models'DATA_DIR.mkdir(parents=True, exist_ok=True)REPORTS_DIR.mkdir(parents=True, exist_ok=True)MODELS_DIR.mkdir(parents=True, exist_ok=True)MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', str(REPORTS_DIR / 'mlruns'))mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)mlflow.set_experiment('f2b_automl_lazypredict')print(f"=== Environment Info ===")print(f"IN_COLAB: {IN_COLAB}")print(f"HAS_CUDA: {HAS_CUDA}")print(f"HAS_CUDF: {HAS_CUDF}")print(f"HAS_TORCH: {HAS_TORCH}")if HAS_CUDA:    print(f"GPU Memory: {GPU_MEMORY / (1024**3):.1f} GB")if TOTAL_MEMORY:    print(f"System RAM: {TOTAL_MEMORY / (1024**3):.1f} GB")print(f"BASE: {BASE}")print(f"DATA_DIR: {DATA_DIR}")print(f"MLFLOW_TRACKING_URI: {MLFLOW_TRACKING_URI}")

## Carga de DatosDataset canónico desde `f1_eda_definitivo`. Se mapea `target_class` (Negativo/Neutro/Positivo) a valores numéricos con LabelEncoder.

In [ ]:
print('Cargando dataset canonico desde f1_eda_definitivo...')CANONICAL_PATH = DATA_DIR / 'office_products_balanced.parquet'try:    df = pd.read_parquet(CANONICAL_PATH, columns=['text', 'rating', 'target_class'])except FileNotFoundError:    print(f'[ERROR] Dataset canonico no encontrado en {CANONICAL_PATH}')    print('Ejecute primero f1_eda_definitivo.ipynb en Colab para generarlo.')    raiseexcept Exception as e:    print(f'[ERROR] No se pudo cargar el dataset: {e}')    raiselabel_encoder = LabelEncoder()df['target'] = label_encoder.fit_transform(df['target_class'])print('Mapeo de clases:', dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))# word_count filterdf['word_count'] = df['text'].astype(str).str.split().str.len()df = df[df['word_count'] >= 5]print(f'Registros tras filtro word_count >= 5: {len(df):,}')print('\nDistribucion de target (0=Negativo, 1=Neutro, 2=Positivo):')print(df['target'].value_counts().sort_index())SUBSAMPLE_SIZE = 1_000_000print(f'\nSubmuestreo estratificado a {SUBSAMPLE_SIZE:,} filas...')df, _ = train_test_split(    df, train_size=SUBSAMPLE_SIZE, random_state=RANDOM_SEED, stratify=df['target'])print(f'Dataset reducido a {len(df):,} filas')print(df['target'].value_counts().sort_index())

## Submuestreo EstratificadoLazyPredict ejecuta 27+ clasificadores en CPU, lo que requiere reducir el volumen para evitar OOM. Se toma una muestra estratificada de ~50K filas (~16.6K por clase). Esto es estadísticamente representativo para el benchmark.

In [ ]:
SAMPLE_SIZE = 50000print(f'Submuestreo estratificado a {SAMPLE_SIZE} filas...')df_sample = df.groupby('target', group_keys=False).apply(    lambda x: x.sample(n=min(SAMPLE_SIZE // 3, len(x)), random_state=RANDOM_SEED)).reset_index(drop=True)print(f'Muestra estratificada: {len(df_sample):,} filas')print(df_sample['target'].value_counts().sort_index())del dfgc.collect()

## División Train/TestSe divide ANTES de vectorizar para prevenir Data Leakage. El vocabulario de test no debe contaminar el vectorizador.

In [ ]:
print('Dividiendo en Train/Test (80/20)...')X_train_text, X_test_text, y_train, y_test = train_test_split(    df_sample['text'], df_sample['target'],    test_size=0.2, random_state=RANDOM_SEED, stratify=df_sample['target'])del df_samplegc.collect()print(f'Train: {len(X_train_text):,} | Test: {len(X_test_text):,}')

## Vectorización TF-IDFSe ajusta el vectorizador solo sobre Train. Test solo se transforma.

In [ ]:
print('Vectorizando con TF-IDF...')vectorizer = TfidfVectorizer(    max_features=1500,    stop_words='english',    ngram_range=(1, 2),    min_df=5)X_train = vectorizer.fit_transform(X_train_text.astype(str))X_test = vectorizer.transform(X_test_text.astype(str))print(f'Matriz TF-IDF: Train {X_train.shape} | Test {X_test.shape}')joblib.dump(vectorizer, MODELS_DIR / 'tfidf_vectorizer_lazypredict.joblib')print(f'Vectorizer guardado en {MODELS_DIR / "tfidf_vectorizer_lazypredict.joblib"}')

## AutoML Benchmark (LazyPredict)Se ejecutan 27+ clasificadores sobre la misma partición. Todos los modelos comparten el mismo vectorizador y partición, garantizando comparación justa.

In [ ]:
print('Ejecutando LazyPredict (27+ clasificadores)...')clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None, class_weight='balanced')with mlflow.start_run(run_name='lazypredict_benchmark'):    mlflow.set_tag('model_type', 'automl_benchmark')    mlflow.log_param('sample_size', SAMPLE_SIZE)    mlflow.log_param('vectorizer_max_features', 1500)    mlflow.log_param('random_state', RANDOM_SEED)    models_df, predictions = clf.fit(X_train, X_test, y_train, y_test)    print('\nTop 10 modelos segun LazyPredict:')    print(models_df.head(10).to_string())    # Log best F1 to MLflow    best_f1 = models_df['F1 Score'].max()    best_model = models_df['F1 Score'].idxmax()    mlflow.log_metric('best_f1_score', best_f1)    mlflow.log_param('best_model', best_model)results_path = REPORTS_DIR / 'lazypredict_results.csv'models_df.to_csv(results_path)print(f'Resultados guardados en {results_path}')gc.collect()

### Visualización: Top 10 Modelos

In [ ]:
top = models_df.head(10).reset_index()fig, ax = plt.subplots(figsize=(12, 6))bars = ax.barh(range(len(top)), top['F1 Score'], color='steelblue')ax.set_yticks(range(len(top)))ax.set_yticklabels(top['Model'])ax.set_xlabel('F1 Score')ax.set_title('Top 10 Modelos - LazyPredict (F1-Score)')ax.invert_yaxis()plt.tight_layout()plt.show()

## Modelo Explicativo: LightGBMUna vez identificado el mejor perfil de modelo con LazyPredict, se entrena LightGBM como modelo explicativo con exportación a Pipeline.

In [ ]:
print('Entrenando LightGBM explicativo...')modelo_explicativo = lgb.LGBMClassifier(    class_weight='balanced', random_state=RANDOM_SEED,    n_jobs=-1, verbose=-1)modelo_explicativo.fit(X_train, y_train)y_pred_lgb = modelo_explicativo.predict(X_test)f1 = f1_score(y_test, y_pred_lgb, average='macro')print(f'LightGBM F1-macro: {f1:.4f}')pipeline_lgb = Pipeline([('tfidf', vectorizer), ('clf', modelo_explicativo)])joblib.dump(pipeline_lgb, MODELS_DIR / 'pipeline_lgb_lazypredict.joblib')with mlflow.start_run(run_name='lightgbm_explicativo'):    mlflow.set_tag('model_type', 'lightgbm')    mlflow.log_param('random_state', RANDOM_SEED)    mlflow.log_param('class_weight', 'balanced')    mlflow.log_metric('f1_macro', f1)print(f'Pipeline LightGBM guardado en {MODELS_DIR / "pipeline_lgb_lazypredict.joblib"}')

### Importancia de Features (LightGBM)

In [ ]:
feature_names = vectorizer.get_feature_names_out()importances = modelo_explicativo.feature_importances_df_importances = pd.DataFrame({    'Caracteristica': feature_names,    'Importancia': importances}).sort_values(by='Importancia', ascending=False).head(20)fig, ax = plt.subplots(figsize=(10, 6))ax.barh(range(len(df_importances)), df_importances['Importancia'].values, color='coral')ax.set_yticks(range(len(df_importances)))ax.set_yticklabels(df_importances['Caracteristica'].values)ax.set_xlabel('Importancia')ax.set_title('Top 20 Features - LightGBM')ax.invert_yaxis()plt.tight_layout()plt.show()

## Resumen

In [ ]:
print('\nBenchmark AutoML completado.')print(f'Resultados: lazypredict_results.csv')print('Compare estos resultados con las metricas en metrics_fase2.json (Fase 2A).')print(f'Mejor F1-score (macro) LazyPredict: {best_f1:.4f} - {best_model}')print(f'LightGBM F1-macro: {f1:.4f}')